# Heart Disease Risk: Supervised Learning, Clustering, and PCA

This notebook presents an end-to-end machine-learning analysis of a **synthetic** heart-disease risk dataset. It covers exploratory data analysis, supervised classification, hyperparameter tuning, error analysis, unsupervised clustering, and dimensionality reduction with PCA.

> **Medical disclaimer:** This project is educational. The data is synthetic and the models must not be used for diagnosis, screening, or medical decision-making.


## Contents

1. Setup and data loading
2. Data quality and exploratory analysis
3. Supervised learning
4. Final model evaluation and error analysis
5. Clustering: K-Means, hierarchical clustering, and GMM
6. Principal component analysis (PCA)
7. Conclusions and limitations


## 1. Setup and Data Loading


In [ ]:
import os
import shutil
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    calinski_harabasz_score,
    classification_report,
    confusion_matrix,
    davies_bouldin_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
TARGET = "has_heart_disease"
ID_COLUMN = "patient_id"
# One job is the most portable default for notebooks and avoids oversubscribing
# constrained environments. Set the N_JOBS environment variable to use more.
N_JOBS = int(os.getenv("N_JOBS", "1"))

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


In [ ]:
def find_project_root() -> Path:
    # Locate the repository whether Jupyter starts in the root or notebooks/.
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "heart_disease_risk_2026.csv"
KAGGLE_DATASET = "uditjain13/heart-disease-risk-2026"

if not DATA_PATH.exists():
    try:
        import kagglehub

        os.environ.setdefault(
            "KAGGLEHUB_CACHE",
            str(PROJECT_ROOT / ".cache" / "kagglehub"),
        )
        downloaded_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        downloaded_file = downloaded_dir / DATA_PATH.name
        if not downloaded_file.exists():
            raise FileNotFoundError(f"Downloaded folder did not contain {DATA_PATH.name}")

        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(downloaded_file, DATA_PATH)
    except Exception as exc:
        raise FileNotFoundError(
            f"Dataset not found at {DATA_PATH}. Download it from Kaggle and place it "
            "at that path, or install/configure kagglehub."
        ) from exc

df = pd.read_csv(DATA_PATH)

expected_columns = {
    "patient_id", "age", "sex", "resting_bp_systolic",
    "resting_bp_diastolic", "cholesterol_total", "hdl", "ldl",
    "triglycerides", "fasting_blood_sugar", "hba1c", "bmi",
    "resting_heart_rate", "max_heart_rate_achieved", "chest_pain_type",
    "exercise_induced_angina", "st_depression", "family_history",
    "smoker_status", "alcohol_units_per_week", "exercise_minutes_per_week",
    "sleep_hours", "stress_score", "wearable_owner", "daily_steps",
    "diet_quality_score", "has_heart_disease",
}
missing_columns = sorted(expected_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

print(f"Loaded: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


## 2. Data Quality and Exploratory Analysis


In [ ]:
quality_summary = pd.DataFrame(
    {
        "Measure": [
            "Rows",
            "Columns",
            "Duplicate rows",
            "Missing values",
            "Positive target rate",
        ],
        "Value": [
            f"{len(df):,}",
            df.shape[1],
            int(df.duplicated().sum()),
            int(df.isna().sum().sum()),
            f"{df[TARGET].mean():.1%}",
        ],
    }
)

display(quality_summary)
display(df.dtypes.rename("dtype").to_frame().T)
display(df.describe().T)

assert df[TARGET].isin([0, 1]).all(), "Target must contain only 0 and 1."
assert df[ID_COLUMN].is_unique, "patient_id must be unique."


In [ ]:
categorical_columns_eda = df.select_dtypes(include=["object", "category", "bool"]).columns

category_summary = pd.DataFrame(
    {
        "Column": categorical_columns_eda,
        "Unique values": [df[column].nunique() for column in categorical_columns_eda],
        "Values": [", ".join(map(str, df[column].dropna().unique())) for column in categorical_columns_eda],
    }
)
display(category_summary)


In [ ]:
numeric_for_correlation = df.select_dtypes(include=np.number).drop(columns=ID_COLUMN)
correlation_matrix = numeric_for_correlation.corr()

plt.figure(figsize=(17, 13))
sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.3,
    fmt=".2f",
)
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

sns.countplot(data=df, x=TARGET, ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set_title("Heart Disease Distribution")
axes[0, 0].set_xlabel("Heart Disease (0 = No, 1 = Yes)")

sns.histplot(data=df, x="age", bins=20, kde=True, ax=axes[0, 1], color="#4C78A8")
axes[0, 1].set_title("Age Distribution")

sns.boxplot(data=df, x=TARGET, y="age", ax=axes[0, 2], color="#72B7B2")
axes[0, 2].set_title("Age by Heart-Disease Status")

sns.countplot(data=df, x="sex", hue=TARGET, ax=axes[1, 0])
axes[1, 0].set_title("Heart Disease by Sex")

sns.countplot(data=df, x="chest_pain_type", hue=TARGET, ax=axes[1, 1])
axes[1, 1].set_title("Heart Disease by Chest-Pain Type")
axes[1, 1].tick_params(axis="x", rotation=20)

sns.boxplot(data=df, x=TARGET, y="cholesterol_total", ax=axes[1, 2], color="#F58518")
axes[1, 2].set_title("Total Cholesterol by Target")

sns.scatterplot(
    data=df,
    x="age",
    y="max_heart_rate_achieved",
    hue=TARGET,
    alpha=0.45,
    ax=axes[2, 0],
)
axes[2, 0].set_title("Age vs Maximum Heart Rate")

sns.boxplot(data=df, x=TARGET, y="resting_bp_systolic", ax=axes[2, 1], color="#E45756")
axes[2, 1].set_title("Systolic Blood Pressure by Target")

sns.boxplot(data=df, x=TARGET, y="stress_score", ax=axes[2, 2], color="#54A24B")
axes[2, 2].set_title("Stress Score by Target")

for axis in axes.flat:
    axis.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
target_correlations = (
    correlation_matrix[TARGET]
    .drop(TARGET)
    .sort_values()
)

plt.figure(figsize=(9, 7))
target_correlations.plot(kind="barh", color="#4C78A8")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Numerical Feature Correlation with Heart Disease")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()


## 3. Supervised Learning

The target is binary. `patient_id` is excluded because it is an identifier, not a medical predictor. The split is stratified to preserve the class ratio, and all encoding/scaling is performed inside pipelines to prevent leakage during cross-validation.


In [ ]:
X = df.drop(columns=[TARGET, ID_COLUMN])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
binary_columns = X.select_dtypes(include="bool").columns.tolist()

split_summary = pd.DataFrame(
    {
        "Split": ["Training", "Testing"],
        "Rows": [len(X_train), len(X_test)],
        "Positive rate": [y_train.mean(), y_test.mean()],
    }
)
display(split_summary)
print(f"Numerical: {len(numeric_columns)} | Categorical: {len(categorical_columns)} | Boolean: {len(binary_columns)}")


In [ ]:
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
        ("binary", "passthrough", binary_columns),
    ],
    verbose_feature_names_out=False,
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
        ("binary", "passthrough", binary_columns),
    ],
    verbose_feature_names_out=False,
)

models = {
    "Logistic Regression": Pipeline(
        [
            ("preprocess", clone(scaled_preprocessor)),
            ("model", LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
        ]
    ),
    "SVM": Pipeline(
        [
            ("preprocess", clone(scaled_preprocessor)),
            ("model", SVC(kernel="rbf", random_state=RANDOM_STATE)),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("preprocess", clone(tree_preprocessor)),
            ("model", RandomForestClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=RANDOM_STATE)),
        ]
    ),
    "Gradient Boosting": Pipeline(
        [
            ("preprocess", clone(tree_preprocessor)),
            ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}


In [ ]:
initial_results = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=N_JOBS,
    )
    initial_results.append(
        {
            "Model": name,
            "Train F1": scores["train_f1"].mean(),
            "CV Accuracy": scores["test_accuracy"].mean(),
            "CV Precision": scores["test_precision"].mean(),
            "CV Recall": scores["test_recall"].mean(),
            "CV F1": scores["test_f1"].mean(),
            "F1 Gap": scores["train_f1"].mean() - scores["test_f1"].mean(),
            "CV F1 Std": scores["test_f1"].std(),
            "CV ROC-AUC": scores["test_roc_auc"].mean(),
        }
    )

initial_comparison = (
    pd.DataFrame(initial_results)
    .sort_values("CV F1", ascending=False)
    .reset_index(drop=True)
)
display(initial_comparison.round(4))


### Hyperparameter Tuning

The searches use only the training set and the same five stratified folds. F1 is the optimization metric because the target is moderately imbalanced and both precision and recall matter.


In [ ]:
searches = {
    "Logistic Regression": GridSearchCV(
        estimator=models["Logistic Regression"],
        param_grid={
            "model__C": [0.01, 0.1, 1, 10],
            "model__class_weight": [None, "balanced"],
        },
        scoring="f1",
        cv=cv,
        n_jobs=N_JOBS,
        return_train_score=True,
    ),
    "SVM": RandomizedSearchCV(
        estimator=models["SVM"],
        param_distributions=[
            {
                "model__kernel": ["linear"],
                "model__C": [0.1, 1, 10],
                "model__class_weight": [None, "balanced"],
            },
            {
                "model__kernel": ["rbf"],
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", 0.01, 0.1, 1],
                "model__class_weight": [None, "balanced"],
            },
        ],
        n_iter=10,
        scoring="f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        return_train_score=True,
    ),
    "Random Forest": RandomizedSearchCV(
        estimator=models["Random Forest"],
        param_distributions={
            "model__n_estimators": [200, 300, 500],
            "model__max_depth": [5, 10, 15, 20, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 5, 10],
            "model__max_features": ["sqrt", "log2"],
            "model__class_weight": [None, "balanced"],
        },
        n_iter=12,
        scoring="f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        return_train_score=True,
    ),
    "Gradient Boosting": RandomizedSearchCV(
        estimator=models["Gradient Boosting"],
        param_distributions={
            "model__n_estimators": [50, 100, 150, 200, 300],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
            "model__max_depth": [1, 2, 3, 4],
            "model__min_samples_leaf": [1, 5, 10, 20],
            "model__subsample": [0.7, 0.8, 1.0],
        },
        n_iter=12,
        scoring="f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        return_train_score=True,
    ),
}

best_models = {}
for name, search in searches.items():
    print(f"Tuning {name}...")
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"  Best CV F1: {search.best_score_:.4f}")
    print(f"  Parameters: {search.best_params_}")


In [ ]:
tuned_results = []

for name, model in best_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=N_JOBS,
    )
    tuned_results.append(
        {
            "Model": name,
            "Train F1": scores["train_f1"].mean(),
            "CV Accuracy": scores["test_accuracy"].mean(),
            "CV Precision": scores["test_precision"].mean(),
            "CV Recall": scores["test_recall"].mean(),
            "CV F1": scores["test_f1"].mean(),
            "F1 Gap": scores["train_f1"].mean() - scores["test_f1"].mean(),
            "CV F1 Std": scores["test_f1"].std(),
            "CV ROC-AUC": scores["test_roc_auc"].mean(),
            "Best Parameters": searches[name].best_params_,
        }
    )

tuned_comparison = (
    pd.DataFrame(tuned_results)
    .sort_values(["CV F1", "CV ROC-AUC"], ascending=False)
    .reset_index(drop=True)
)
display(tuned_comparison)


## 4. Final Model Evaluation and Error Analysis


In [ ]:
final_model_name = tuned_comparison.loc[0, "Model"]
final_model = best_models[final_model_name]
final_model.fit(X_train, y_train)

test_predictions = final_model.predict(X_test)
if hasattr(final_model, "predict_proba"):
    test_scores = final_model.predict_proba(X_test)[:, 1]
else:
    test_scores = final_model.decision_function(X_test)

test_metrics = pd.DataFrame(
    {
        "Model": [final_model_name],
        "Accuracy": [accuracy_score(y_test, test_predictions)],
        "Precision": [precision_score(y_test, test_predictions, zero_division=0)],
        "Recall": [recall_score(y_test, test_predictions, zero_division=0)],
        "F1": [f1_score(y_test, test_predictions, zero_division=0)],
        "ROC-AUC": [roc_auc_score(y_test, test_scores)],
    }
)

print(f"Selected model: {final_model_name}")
display(test_metrics.round(4))
print(classification_report(y_test, test_predictions, target_names=["No Heart Disease", "Heart Disease"]))


In [ ]:
cm = confusion_matrix(y_test, test_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"],
)
plt.title(f"Confusion Matrix — {final_model_name}")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()


In [ ]:
error_analysis = X_test.copy()
error_analysis["Actual"] = y_test
error_analysis["Predicted"] = test_predictions
error_analysis["Prediction Score"] = test_scores

conditions = [
    (error_analysis["Actual"] == 1) & (error_analysis["Predicted"] == 1),
    (error_analysis["Actual"] == 0) & (error_analysis["Predicted"] == 0),
    (error_analysis["Actual"] == 0) & (error_analysis["Predicted"] == 1),
    (error_analysis["Actual"] == 1) & (error_analysis["Predicted"] == 0),
]
error_analysis["Result Type"] = np.select(
    conditions,
    ["True Positive", "True Negative", "False Positive", "False Negative"],
    default="Unknown",
)

error_counts = error_analysis["Result Type"].value_counts().reindex(
    ["True Positive", "True Negative", "False Positive", "False Negative"]
)
display(error_counts.rename("Count").to_frame())

analysis_features = [
    "age",
    "resting_bp_systolic",
    "fasting_blood_sugar",
    "hba1c",
    "bmi",
    "max_heart_rate_achieved",
    "st_depression",
    "Prediction Score",
]
display(error_analysis.groupby("Result Type")[analysis_features].mean().round(2))


False negatives deserve particular attention in this educational setting because they represent affected cases predicted as negative. In a real clinical workflow, threshold selection, probability calibration, external validation, and domain review would all be required.


## 5. Unsupervised Learning: Clustering

The target and patient identifier are excluded from clustering. The target is used only afterward to describe the discovered groups; it does not influence cluster formation.


In [ ]:
clustering_df = df.drop(columns=[TARGET, ID_COLUMN])

cluster_numeric_columns = clustering_df.select_dtypes(include=np.number).columns.tolist()
cluster_categorical_columns = clustering_df.select_dtypes(include=["object", "category"]).columns.tolist()
cluster_binary_columns = clustering_df.select_dtypes(include="bool").columns.tolist()

cluster_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), cluster_numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cluster_categorical_columns),
        ("binary", "passthrough", cluster_binary_columns),
    ],
    verbose_feature_names_out=False,
)

X_cluster = cluster_preprocessor.fit_transform(clustering_df)
print(f"Clustering matrix: {X_cluster.shape[0]:,} rows × {X_cluster.shape[1]} encoded features")


### K-Means: Selecting the Number of Clusters


In [ ]:
k_values = range(2, 11)
kmeans_selection = []

for k in k_values:
    candidate = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = candidate.fit_predict(X_cluster)
    kmeans_selection.append(
        {
            "k": k,
            "Inertia": candidate.inertia_,
            "Silhouette": silhouette_score(
                X_cluster,
                labels,
                sample_size=3_000,
                random_state=RANDOM_STATE,
            ),
        }
    )

kmeans_selection = pd.DataFrame(kmeans_selection)
best_k = int(kmeans_selection.loc[kmeans_selection["Silhouette"].idxmax(), "k"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(kmeans_selection["k"], kmeans_selection["Inertia"], marker="o")
axes[0].set(title="K-Means Elbow Method", xlabel="Number of clusters (k)", ylabel="Inertia")
axes[1].plot(kmeans_selection["k"], kmeans_selection["Silhouette"], marker="o", color="#F58518")
axes[1].set(title="K-Means Silhouette Score", xlabel="Number of clusters (k)", ylabel="Silhouette")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

display(kmeans_selection.round({"Inertia": 2, "Silhouette": 4}))
print(f"Best k by sampled silhouette score: {best_k}")


In [ ]:
kmeans_model = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
kmeans_labels = kmeans_model.fit_predict(X_cluster)

hierarchical_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hierarchical_labels = hierarchical_model.fit_predict(X_cluster)

# A sample keeps the dendrogram readable and avoids plotting 9,000 leaves.
rng = np.random.default_rng(RANDOM_STATE)
dendrogram_indices = rng.choice(len(X_cluster), size=750, replace=False)
linked_sample = linkage(X_cluster[dendrogram_indices], method="ward")

plt.figure(figsize=(14, 6))
dendrogram(linked_sample, no_labels=True, color_threshold=None)
plt.title("Hierarchical Clustering Dendrogram (750-Row Sample)")
plt.xlabel("Sampled observations")
plt.ylabel("Ward distance")
plt.tight_layout()
plt.show()


### Gaussian Mixture Model (GMM)


In [ ]:
component_values = range(2, 11)
gmm_selection = []

for components in component_values:
    candidate = GaussianMixture(
        n_components=components,
        covariance_type="full",
        n_init=2,
        max_iter=300,
        reg_covar=1e-6,
        random_state=RANDOM_STATE,
    )
    labels = candidate.fit_predict(X_cluster)
    gmm_selection.append(
        {
            "Components": components,
            "BIC": candidate.bic(X_cluster),
            "AIC": candidate.aic(X_cluster),
            "Silhouette": silhouette_score(
                X_cluster,
                labels,
                sample_size=3_000,
                random_state=RANDOM_STATE,
            ),
        }
    )

gmm_selection = pd.DataFrame(gmm_selection)
best_gmm_components = int(
    gmm_selection.loc[gmm_selection["Silhouette"].idxmax(), "Components"]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(gmm_selection["Components"], gmm_selection["BIC"], marker="o", label="BIC")
axes[0].plot(gmm_selection["Components"], gmm_selection["AIC"], marker="o", label="AIC")
axes[0].set(title="GMM Information Criteria", xlabel="Components", ylabel="Score (lower is better)")
axes[0].legend()
axes[1].plot(gmm_selection["Components"], gmm_selection["Silhouette"], marker="o", color="#F58518")
axes[1].set(title="GMM Silhouette Score", xlabel="Components", ylabel="Silhouette")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

display(gmm_selection.round(4))
print(f"Components selected by sampled silhouette score: {best_gmm_components}")


In [ ]:
gmm_model = GaussianMixture(
    n_components=best_gmm_components,
    covariance_type="full",
    n_init=2,
    max_iter=300,
    reg_covar=1e-6,
    random_state=RANDOM_STATE,
)
gmm_labels = gmm_model.fit_predict(X_cluster)


def clustering_metrics(name, matrix, labels):
    return {
        "Algorithm": name,
        "Clusters": len(np.unique(labels)),
        "Silhouette ↑": silhouette_score(matrix, labels),
        "Davies-Bouldin ↓": davies_bouldin_score(matrix, labels),
        "Calinski-Harabasz ↑": calinski_harabasz_score(matrix, labels),
    }


cluster_metrics = pd.DataFrame(
    [
        clustering_metrics("K-Means", X_cluster, kmeans_labels),
        clustering_metrics("Hierarchical", X_cluster, hierarchical_labels),
        clustering_metrics("GMM", X_cluster, gmm_labels),
    ]
).sort_values("Silhouette ↑", ascending=False)

display(cluster_metrics.round(4))


In [ ]:
clustered_df = df.copy()
clustered_df["kmeans_cluster"] = kmeans_labels
clustered_df["hierarchical_cluster"] = hierarchical_labels
clustered_df["gmm_cluster"] = gmm_labels

profile_features = [
    "age",
    "bmi",
    "cholesterol_total",
    "resting_heart_rate",
    "max_heart_rate_achieved",
    "hba1c",
    "daily_steps",
    "sleep_hours",
    "stress_score",
]

print("K-Means cluster sizes")
display(clustered_df["kmeans_cluster"].value_counts().sort_index().rename("Rows").to_frame())
print("K-Means numerical profiles")
display(clustered_df.groupby("kmeans_cluster")[profile_features].mean().round(2))
print("Target distribution by K-Means cluster (post-hoc only)")
display(pd.crosstab(clustered_df["kmeans_cluster"], clustered_df[TARGET], normalize="index").round(3))


### Reading the Clustering Metrics

- **Silhouette:** higher is better; values near 0 indicate overlapping clusters.
- **Davies–Bouldin:** lower is better.
- **Calinski–Harabasz:** higher is better, but it is most meaningful when comparing alternatives on the same data.

The metrics should be considered together with cluster stability and domain usefulness—not as universal pass/fail thresholds.


## 6. Principal Component Analysis (PCA)


In [ ]:
pca_all = PCA().fit(X_cluster)
cumulative_variance = np.cumsum(pca_all.explained_variance_ratio_)
n_components_95 = int(np.argmax(cumulative_variance >= 0.95) + 1)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.95, color="red", linestyle="--", label="95% variance")
plt.axvline(n_components_95, color="gray", linestyle=":", label=f"{n_components_95} components")
plt.title("PCA Cumulative Explained Variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

pca_reducer = PCA(n_components=0.95, svd_solver="full")
X_pca = pca_reducer.fit_transform(X_cluster)

print(f"Original encoded features: {X_cluster.shape[1]}")
print(f"Components retained: {X_pca.shape[1]}")
print(f"Variance preserved: {pca_reducer.explained_variance_ratio_.sum():.2%}")


The 95% PCA matrix is the actual reduced dataset used for the before/after comparison. A separate two-component projection is created only for visualization; it is not presented as retaining 95% of the information.


In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_cluster)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
visualizations = [
    ("K-Means", kmeans_labels),
    ("Hierarchical", hierarchical_labels),
    ("GMM", gmm_labels),
]

for axis, (title, labels) in zip(axes, visualizations):
    scatter = axis.scatter(
        X_pca_2d[:, 0],
        X_pca_2d[:, 1],
        c=labels,
        cmap="viridis",
        alpha=0.45,
        s=12,
    )
    axis.set_title(f"{title} Clusters")
    axis.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})")
    axis.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})")
    axis.grid(alpha=0.2)

plt.suptitle("Two-Component PCA Projection (Visualization Only)", y=1.03, fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
kmeans_before_pca = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
labels_before_pca = kmeans_before_pca.fit_predict(X_cluster)

kmeans_after_pca = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
labels_after_pca = kmeans_after_pca.fit_predict(X_pca)

pca_comparison = pd.DataFrame(
    {
        "Metric": ["Silhouette ↑", "Davies-Bouldin ↓", "Calinski-Harabasz ↑"],
        "Before PCA": [
            silhouette_score(X_cluster, labels_before_pca),
            davies_bouldin_score(X_cluster, labels_before_pca),
            calinski_harabasz_score(X_cluster, labels_before_pca),
        ],
        "After PCA": [
            silhouette_score(X_pca, labels_after_pca),
            davies_bouldin_score(X_pca, labels_after_pca),
            calinski_harabasz_score(X_pca, labels_after_pca),
        ],
    }
)

display(pca_comparison.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, row in zip(axes, pca_comparison.itertuples(index=False)):
    bars = axis.bar(["Before PCA", "After PCA"], [row[1], row[2]], color=["#4C78A8", "#F58518"])
    axis.set_title(row[0])
    axis.bar_label(bars, fmt="%.3f", padding=3)
    axis.grid(axis="y", alpha=0.2)

plt.suptitle("K-Means Performance Before and After 95% PCA", y=1.03, fontsize=15)
plt.tight_layout()
plt.show()


## 7. Conclusions and Limitations

- The supervised pipelines prevent preprocessing leakage and compare four different model families under the same stratified cross-validation scheme.
- The final classifier is selected using training-set cross-validation, then evaluated once on the untouched test set.
- Clustering quality is modest when silhouette values are close to zero, so the clusters should be treated as exploratory segments rather than clearly separated natural groups.
- PCA reduces the encoded feature space while preserving at least 95% of total variance. The two-component figures are visual summaries only.
- The dataset is synthetic, lacks external clinical validation, and may not represent real patient populations. No result in this notebook should be interpreted as medical advice or a deployable diagnostic system.
